# 03 — Activity Over Time

Monthly task starts, completions, status composition, and category trends.


In [ ]:
%run pathutils.ipynb
%run database.ipynb
%run export.ipynb

import sys
from datetime import date
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, str(Path(get_project_root_folder()) / "reports"))
from reporting import grouped_summary, monthly_activity, normalise_history, open_task_analysis, percentage, summary


In [ ]:
# Inclusive reporting period, based on task start date.
START_DATE = "2024-01-01"
END_DATE = "9999-12-31"
REPORT_DATE = date.today()

query = construct_query("task-history.sql", {"START-DATE": START_DATE, "END-DATE": END_DATE})
history = normalise_history(query_data(query))
period_label = f"{START_DATE} to {END_DATE} ({len(history):,} tasks)"
print(f"Reporting period: {period_label}")


## Monthly activity

In [ ]:
activity = monthly_activity(history)
activity

In [ ]:
activity.plot(x="Month", y=["Tasks Started", "Tasks Completed"], marker="o", title=f"Monthly activity — {period_label}")
plt.ylabel("Tasks"); plt.xlabel("Month"); plt.tight_layout(); plt.show()

## Category composition by start month

In [ ]:
category_trends = history.pivot_table(index="Start Month", columns="Category", values="Task ID", aggfunc="count", fill_value=0).reset_index()
category_trends

In [ ]:
category_trends.set_index("Start Month").plot.area(stacked=True, title=f"Category composition by month — {period_label}")
plt.ylabel("Tasks started"); plt.xlabel("Month"); plt.tight_layout(); plt.show()

In [ ]:
EXPORT_NAME = "03-activity-over-time.xlsx"
EXPORT_DATA = {"Monthly Activity": activity, "Category Trends": category_trends}
export_to_spreadsheet(
    get_export_folder_path(), EXPORT_NAME, EXPORT_DATA
)
print(f"Exported {EXPORT_NAME} to {get_export_folder_path()}")
